## Variational Diffusion Models
In this notebook, I will attempt to relate the main contents of the paper to the PyTorch implementation.  
This is not expected to be a run-able notebook but rather connect everything conceptually.  
I will add caveats for things to add or points of confusion.

Code taken from here: https://github.com/addtt/variational-diffusion-models/blob/main/README.md 

### Forward process
Learned noise schedule based on signal to noise ratio with variance preservation.  
- While this can be learned, we will not be completing variance minimisation and will instead have these as set hyperparameters.   

Each step adds increasing amounts of noise according to the noise schedule.  


In [ ]:
class VDM(nn.Module): ## All code following lives under this class
    def __init__(self, model, cfg, image_shape):
        super().__init__()
        self.model = model
        # self.cfg = cfg  # not needed? always use fixedlinear schedule
        self.image_shape = image_shape
        self.vocab_size = 256
        self.gamma = FixedLinearSchedule(cfg.gamma_min, cfg.gamma_max)

In [ ]:

## Noise schedule with fixed linear interpolation between min and max gamma values.
class FixedLinearSchedule(nn.Module):
    def __init__(self, gamma_min, gamma_max):
        super().__init__()
        self.gamma_min = gamma_min
        self.gamma_max = gamma_max

    def forward(self, t):
        return self.gamma_min + (self.gamma_max - self.gamma_min) * t

#### Forward definition split up by tasks
First task is CIFAR10 image formatting

In [ ]:
def forward(self, batch, *, noise=None):

In [ ]:
        # Safety and processing checks of input batch.
        # x, label = maybe_unpack_batch(batch)
        # assert x.shape[1:] == self.image_shape
        # assert 0.0 <= x.min() and x.max() <= 1.0

        bpd_factor = 1 / (np.prod(x.shape[1:]) * np.log(2)) ### determine bpd factor based on image shape to normalise

        # Convert image to integers in range [0, vocab_size - 1].
        img_int = torch.round(x * (self.vocab_size - 1)).long() 
        
        # vocab size for CIFAR 10 is 256 as there are 256 unique pixel values
        

        # SANITY CHECKS
        # assert (img_int >= 0).all() and (img_int <= self.vocab_size - 1).all()
        # # Check that the image was discrete with vocab_size values.
        # assert allclose(img_int / (self.vocab_size - 1), x)

        # Rescale integer image to [-1 + 1/vocab_size, 1 - 1/vocab_size]
        x = 2 * ((img_int + 0.5) / self.vocab_size) - 1


        #### IMAGE FORMATTING COMPLETE - NOW IN RANGE [-1, 1] ####

#### Sampling from q(x_t | x_0)
Returns x_t = mean + noise * scale → noisy version of x_0 (the original image) at timestep t  
sigmoid(-gamma_t) = 𝛼_𝑡 → fraction of signal remaining

sigmoid(gamma_t) = 1−𝛼_𝑡 → fraction of noise

mean = attenuated original signal  
scale = standard deviation of the added noise  

In diffusion models (and VDMs), t is randomly sampled during training because the model needs to learn to denoise data from any point in the diffusion process, not just a fixed noise level.

In [ ]:
    def sample_q_t_0(self, x, times, noise=None):
        """Samples from the distributions q(x_t | x_0) at the given time steps."""
        with torch.enable_grad():  # Need gradient to compute loss even when evaluating
            gamma_t = self.gamma(times)
        gamma_t_padded = unsqueeze_right(gamma_t, x.ndim - gamma_t.ndim)


        mean = x * sqrt(sigmoid(-gamma_t_padded))  # x * alpha
        scale = sqrt(sigmoid(gamma_t_padded))

        # # sample Gaussian noise - rendundant in this case as noise is typically provided
        # if noise is None:
        #     noise = torch.randn_like(x) 


        return mean + noise * scale, gamma_t

In [ ]:
    def sample_times(self, batch_size):
        if self.cfg.antithetic_time_sampling:
            t0 = np.random.uniform(0, 1 / batch_size)
            times = torch.arange(t0, 1.0, 1.0 / batch_size, device=self.device)
        # else:
        #     times = torch.rand(batch_size, device=self.device)
        return times

In [ ]:
        # Sample from q(x_t | x_0) with random t.
        times = self.sample_times(x.shape[0]).requires_grad_(True) # time is randomly sampled 
        if noise is None:
            noise = torch.randn_like(x)
        x_t, gamma_t = self.sample_q_t_0(x=x, times=times, noise=noise)

##### Note on antithetic sampling
Antithetic time sampling is a variance reduction technique in Monte Carlo simulations that pairs random samples to create a negative correlation between them, which helps to decrease the variance of the estimate. 

#### Put into VDM vvvv
The model_out is the updated image produced by the "decoder" U net network.

In [ ]:
        # Forward through model
        model_out = self.model(x_t, gamma_t)

### Reverse Process (U Net)

In [ ]:
class UNetVDM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        ### HYPER PARAMETERS ###
        attention_params = dict(
            n_heads=cfg.n_attention_heads,
            n_channels=cfg.embedding_dim,
            norm_groups=cfg.norm_groups,
        )

        resnet_params = dict(
            ch_in=cfg.embedding_dim,
            ch_out=cfg.embedding_dim,
            condition_dim=4 * cfg.embedding_dim,
            dropout_prob=cfg.dropout_prob,
            norm_groups=cfg.norm_groups,
        )

#### Fourier Features
Since VDM aims to optimize for likelihood, which is sensitive to fine scale details and exact values of individual pixels, fourier helps capture these details.
- Improves noise modelling as it is simpler in the frequency domain
- Creates stronger priors in U-net architecture
- Preserve finer details leading to less blurring, etc

Here we apply it to color channels for single pixels, in order to model fine distributional details at the
level of each scalar input.


In [ ]:
        self.fourier_features = FourierFeatures()

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, first=5.0, last=6.0, step=1.0):
        super().__init__()
        self.freqs_exponent = torch.arange(first, last + 1e-8, step)

    @property
    def num_features(self):
        return len(self.freqs_exponent) * 2

    def forward(self, x):
        assert len(x.shape) >= 2

        # Compute (2pi * 2^n) for n in freqs.
        freqs_exponent = self.freqs_exponent.to(dtype=x.dtype, device=x.device)  # (F, )
        freqs = 2.0**freqs_exponent * 2 * pi  # (F, )
        freqs = freqs.view(-1, *([1] * (x.dim() - 1)))  # (F, 1, 1, ...)

        # Compute (2pi * 2^n * x) for n in freqs.
        features = freqs * x.unsqueeze(1)  # (B, F, X1, X2, ...)
        features = features.flatten(1, 2)  # (B, F * C, X1, X2, ...)

        # Output features are cos and sin of above. Shape (B, 2 * F * C, H, W).
        return torch.cat([features.sin(), features.cos()], dim=1)
    


# Helper function to concatenate fourier features
# def maybe_concat_fourier(self, z):
#     return torch.cat([z, self.fourier_features(z)], dim=1)

#### Time embedding and conditioning
A small MLP that converts the scalar diffusion time into a neural “conditioning signal” that guides denoising.

In [ ]:

        self.embed_conditioning = nn.Sequential(
            nn.Linear(cfg.embedding_dim, cfg.embedding_dim * 4),
            nn.SiLU(),
            nn.Linear(cfg.embedding_dim * 4, cfg.embedding_dim * 4),
            nn.SiLU(),
        )

##### Time embedding in more depth
Diffusion models need the UNet to know how much noise is currently in the image.
A single scalar 𝑡 is not expressive enough.  
So we map each scalar 𝑡to a high-dimensional periodic embedding vector that:
- spreads the information across frequency scales
- allows the network to distinguish early vs late timesteps
- gives smooth, continuous encoding of time

In [ ]:
def get_timestep_embedding(
    timesteps,
    embedding_dim: int,
    dtype=torch.float32,
    max_timescale=10_000,
    min_timescale=1,
):
    # Adapted from tensor2tensor and VDM codebase.
    ### SANITY ###
    # assert timesteps.ndim == 1
    # assert embedding_dim % 2 == 0

    timesteps *= 1000.0  # In DDPM the time step is in [0, 1000], here [0, 1]

    ## This creates a set of frequencies from low (slow variation) to high (rapid variation).
    num_timescales = embedding_dim // 2
    inv_timescales = torch.logspace(  # or exp(-linspace(log(min), log(max), n))
        -np.log10(min_timescale),
        -np.log10(max_timescale),
        num_timescales,
        device=timesteps.device,
    )
    emb = timesteps.to(dtype)[:, None] * inv_timescales[None, :]  # (T, D/2)
    return torch.cat([emb.sin(), emb.cos()], dim=1)  # (T, D)

#### Prepare U-Net structure
##### Input
Start with RGB channels  
(1 + num_fourier_features)	Add spatial Fourier features for each channel  
Conv2d(..., cfg.embedding_dim, ...)	Project to UNet working dimension  

In [ ]:
        total_input_ch = cfg.input_channels
        total_input_ch *= 1 + self.fourier_features.num_features
        self.conv_in = nn.Conv2d(total_input_ch, cfg.embedding_dim, 3, padding=1)

##### ResNet Architecture Explained
Learns how much noise removal to apply at this stage of denoising
At each spatial resolution level the model does:
1. Look at features
2. Know how much noise to remove (via condition)
3. Apply denoising appropriate for that timestep
4. Preserve identity path (residual skip)

In [ ]:
class ResnetBlock(nn.Module):
    def __init__(
        self,
        ch_in,
        ch_out=None,
        condition_dim=None,
        dropout_prob=0.0,
        norm_groups=32,
    ):
        super().__init__()
        ch_out = ch_in if ch_out is None else ch_out
        self.ch_out = ch_out
        self.condition_dim = condition_dim

        ### Standard residual pre-activation block (used in DDPMs, Stable Diffusion, VDM). ###
        # Produces initial processed features
        self.net1 = nn.Sequential(
            nn.GroupNorm(num_groups=norm_groups, num_channels=ch_in),
            nn.SiLU(),
            nn.Conv2d(ch_in, ch_out, kernel_size=3, padding=1),
        )

        #### Time embedding dimension projection ####
        ### So the block learns something like: Early timesteps → coarse denoising
        ### Late timesteps → fine detail reconstruction
        if condition_dim is not None:
            self.cond_proj = zero_init(nn.Linear(condition_dim, ch_out, bias=False))

        ### The final conv is zero-initialized — meaning block starts as an identity.
        # This stabilizes diffusion training (same trick as DDPM and Stable Diffusion)
        self.net2 = nn.Sequential(
            nn.GroupNorm(num_groups=norm_groups, num_channels=ch_out),
            nn.SiLU(),
            *([nn.Dropout(dropout_prob)] * (dropout_prob > 0.0)),
            zero_init(nn.Conv2d(ch_out, ch_out, kernel_size=3, padding=1)),
        )


        ### Skip connection if input and output channels differ ### 
        # Preserves information
        if ch_in != ch_out:
            self.skip_conv = nn.Conv2d(ch_in, ch_out, kernel_size=1)

    def forward(self, x, condition):
        h = self.net1(x)

        # Scale and shift with time embedding
        if condition is not None:
            assert condition.shape == (x.shape[0], self.condition_dim)
            condition = self.cond_proj(condition)
            condition = condition[:, :, None, None]
            h = h + condition
        h = self.net2(h)
        if x.shape[1] != self.ch_out:
            x = self.skip_conv(x)
        assert x.shape == h.shape # sanity check
        return x + h

##### Down path
Use multiple blocks to have higher representational capacity
- no attention applied at the higher layers

In [ ]:
        # Down path: n_blocks blocks with a resnet block and maybe attention.
        self.down_blocks = nn.ModuleList(
            UpDownBlock(
                resnet_block=ResnetBlock(**resnet_params),
                attention_block=AttentionBlock(**attention_params)
                # if cfg.attention_everywhere
                else None,
            )
            for _ in range(cfg.n_blocks)
        )

In [ ]:
class UpDownBlock(nn.Module):
    def __init__(self, resnet_block, attention_block=None):
        super().__init__()
        self.resnet_block = resnet_block
        self.attention_block = attention_block

    def forward(self, x, cond):
        x = self.resnet_block(x, cond)
        # if self.attention_block is not None:
        #     x = self.attention_block(x)
        # no attention used
        return x

#### Middle path (bottom of the U)
This block sees the lowest-resolution, highest-feature representation.
Attention is placed at the bottom because that's where the feature map is smallest, so global relationships can be modeled cheaply and effectively. Fills in global semantics

In [ ]:
        self.mid_resnet_block_1 = ResnetBlock(**resnet_params)
        self.mid_attn_block = AttentionBlock(**attention_params)
        self.mid_resnet_block_2 = ResnetBlock(**resnet_params)

##### Attention 
1. GroupNorm  
- Normalizes feature maps → stabilizes training  
2. 1×1 Conv to 3× channels  
- Splits features into Query (Q), Key (K), Value (V) for multi-head attention  
- Each channel is projected to enable attention computation  
3. Attention layer
- Performs self-attention over spatial positions (H×W)
- Computes weighted sum of values based on similarity between queries and keys
- Captures long-range dependencies across the image
4. Zero-initialized 1×1 Conv
- Projects back to original channel size
- Zero-init ensures residual starts as identity → training is stable
5. Residual connection
- Adds original input back
- Preserves features and allows easy gradient flow

In [ ]:
class AttentionBlock(nn.Module):
    """Self-attention residual block."""

    def __init__(self, n_heads, n_channels, norm_groups):
        super().__init__()
        assert n_channels % n_heads == 0
        self.layers = nn.Sequential(
            nn.GroupNorm(num_groups=norm_groups, num_channels=n_channels),
            nn.Conv2d(n_channels, 3 * n_channels, kernel_size=1),  # (B, 3 * C, H, W)
            Attention(n_heads),
            zero_init(nn.Conv2d(n_channels, n_channels, kernel_size=1)),
        )

    def forward(self, x):
        return self.layers(x) + x

In [ ]:
class Attention(nn.Module):
    """Based on https://github.com/openai/guided-diffusion."""

    def __init__(self, n_heads):
        super().__init__()
        self.n_heads = n_heads

    def forward(self, qkv):
        assert qkv.dim() >= 3, qkv.dim()
        assert qkv.shape[1] % (3 * self.n_heads) == 0
        spatial_dims = qkv.shape[2:]
        qkv = qkv.view(*qkv.shape[:2], -1)  # (B, 3*H*C, T)
        out = attention_inner_heads(qkv, self.n_heads)  # (B, H*C, T)
        return out.view(*out.shape[:2], *spatial_dims)

In [ ]:

def attention_inner_heads(qkv, num_heads):
    """Computes attention with heads inside of qkv in the channel dimension.

    Args:
        qkv: Tensor of shape (B, 3*H*C, T) with Qs, Ks, and Vs, where:
            H = number of heads,
            C = number of channels per head.
        num_heads: number of heads.

    Returns:
        Attention output of shape (B, H*C, T).
    """

    bs, width, length = qkv.shape
    ch = width // (3 * num_heads)

    # Split into (q, k, v) of shape (B, H*C, T).
    q, k, v = qkv.chunk(3, dim=1)

    # Rescale q and k. This makes them contiguous in memory.
    scale = ch ** (-1 / 4)  # scale with 4th root = scaling output by sqrt
    q = q * scale
    k = k * scale

    # Reshape qkv to (B*H, C, T).
    new_shape = (bs * num_heads, ch, length)
    q = q.view(*new_shape)
    k = k.view(*new_shape)
    v = v.reshape(*new_shape)

    # Compute attention.
    weight = einsum("bct,bcs->bts", q, k)  # (B*H, T, T)
    weight = softmax(weight.float(), dim=-1).to(weight.dtype)  # (B*H, T, T)
    out = einsum("bts,bcs->bct", weight, v)  # (B*H, C, T)
    return out.reshape(bs, num_heads * ch, length)  # (B, H*C, T)

#### Up path
The up path reconstructs the final image from the compressed latent representation created in the down path (encoder) + bottleneck (mid) blocks.
It has the key tasks:
1. Fuse skip connections from the down path (to preserve fine details).
2. Refine features at each resolution using ResNet blocks.  
Concatenation doubles the channels, so the ResNet block’s input channels must match.

In [ ]:
        # Up path: n_blocks+1 blocks with a resnet block and maybe attention.
        resnet_params["ch_in"] *= 2  # double input channels due to skip connections
        self.up_blocks = nn.ModuleList(
            UpDownBlock(
                resnet_block=ResnetBlock(**resnet_params),
                attention_block=AttentionBlock(**attention_params)
                # if cfg.attention_everywhere
                else None,
            )
            for _ in range(cfg.n_blocks + 1)
        )

#### Output path
self.conv_out maps the UNet’s internal feature representation back to the original input space, e.g., RGB image 

In [ ]:
        self.conv_out = nn.Sequential(
            nn.GroupNorm(num_groups=cfg.norm_groups, num_channels=cfg.embedding_dim),
            nn.SiLU(),
            zero_init(nn.Conv2d(cfg.embedding_dim, cfg.input_channels, 3, padding=1)),
        )

#### Moving through the U Net 
The UNet takes the current noisy image z and the noise level g_t, processes it through local + global feature blocks (ResNet + Attention), and predicts how to update z to reduce noise at this timestep.

##### During training
Sample a random gamma value from the schedule: 𝑔_𝑡∼Uniform(𝛾_min,𝛾_max)
- This represents a random point along the diffusion timeline.
- You inject noise into the clean image according to g_t to create the noisy input z.
The network is trained to predict the residual / denoised image given z and g_t.

##### During inference/sampling
- Start with pure Gaussian noise (z_1)  
- Iteratively denoise conditioned on g_t
- Produce a high-quality sample (z_0)

In [ ]:
    def forward(self, z, g_t):
        # Get gamma to shape (B, ).
        g_t = g_t.expand(z.shape[0])  # assume shape () or (1,) or (B,)
        assert g_t.shape == (z.shape[0],)
        # Rescale to [0, 1], but only approximately since gamma0 & gamma1 are not fixed.
        t = (g_t - self.cfg.gamma_min) / (self.cfg.gamma_max - self.cfg.gamma_min)
        t_embedding = get_timestep_embedding(t, self.cfg.embedding_dim)
        # We will condition on time embedding.
        cond = self.embed_conditioning(t_embedding)

        h = self.maybe_concat_fourier(z)
        h = self.conv_in(h)  # (B, embedding_dim, H, W)
        hs = []
        for down_block in self.down_blocks:  # n_blocks times
            hs.append(h)
            h = down_block(h, cond)
        hs.append(h)
        h = self.mid_resnet_block_1(h, cond)
        h = self.mid_attn_block(h)
        h = self.mid_resnet_block_2(h, cond)
        for up_block in self.up_blocks:  # n_blocks+1 times
            h = torch.cat([h, hs.pop()], dim=1)
            h = up_block(h, cond)
        prediction = self.conv_out(h)
        assert prediction.shape == z.shape, (prediction.shape, z.shape)
        return prediction + z


### Training
This will need to be adapted to look more similar to what we have worked with previously and to work with WandB. However, the basic premise is the same.  
Done like it always is:

Forward pass: loss, _ = self.diffusion_model(data) → computes all the VDM losses (diffusion, latent, reconstruction).  

Backward pass: self.accelerator.backward(loss) → computes gradients.  

Optimizer step: self.opt.step() → updates weights.  

EMA update: self.ema.update() → maintains smoothed version of the model.  

All of this occurs inside the while self.step < self.train_num_steps loop in Trainer.train().  

In [ ]:
x0 = batch
g_t = random_gamma()
z_t, noise = diffusion.q_sample(x0, g_t) # x_t, gamma_t = self.sample_q_t_0(x=x, times=times, noise=noise)
pred = model(z_t, g_t)
loss = diffusion.loss(pred, noise, g_t)
loss.backward()
optimizer.step()

### Loss
The loss functions are all found in the vdm.py file

#### Diffusion loss
gamma_t = gamma(times) is the log-SNR at each timestep.  

In continuous-time VDM, the loss is scaled by the derivative of the log-SNR w.r.t time: 𝑑𝛾_𝑡/𝑑𝑡  ​

autograd.grad(...) computes this derivative for each sample in the batch:
- Shape: (B,) → one value per batch element  

create_graph=True allows backprop through this derivative, needed for training the network.  

In VDMs, the loss at each time step is weighted by how fast the noise changes (dγ/dt). This gives the continuous-time ELBO (Evidence Lower Bound) in bits per dimension.


model_out is the network’s prediction for the noise  
noise is the actual noise added to x_0 to get x_t.  
(model_out - noise)^2 → squared error for each pixel  
.sum((1,2,3)) → sum over all channels, height, and width → gives per-sample loss (B,) 

Multiply by 0.5 → matches Gaussian log-likelihood formula  
Multiply by gamma_grad → weights loss by rate of noise change in continuous time  
Multiply by bpd_factor → converts loss in nats to bits per dimension, and normalizes by number of pixels/dimensions  

For each batch sample, take the predicted noise vs. actual noise, compute squared error, scale it by how fast noise is changing at this timestep, normalize to bits-per-pixel, and that gives the diffusion loss.


At the final timestep, the fully noised image should look like a standard Gaussian. The latent loss measures how far the model’s distribution of 𝑥_1 is from N(0,1) and adds that to the total bits-per-dimension loss

In [ ]:
        # *** Diffusion loss (bpd)
        gamma_grad = autograd.grad(  # gamma_grad shape: (B, )
            gamma_t,  # (B, )
            times,  # (B, )
            grad_outputs=torch.ones_like(gamma_t),
            create_graph=True,
            retain_graph=True,
        )[0]
        pred_loss = ((model_out - noise) ** 2).sum((1, 2, 3))  # (B, )
        diffusion_loss = 0.5 * pred_loss * gamma_grad * bpd_factor

##### Latent loss
At t=1, this corresponds to the most noisy latent, 𝑥_1​ close to standard Gaussian.  
sigmoid(gamma_1) = 1−𝛼_1 → variance of the Gaussian at t=1.Variance of 1(x_1 | x_0)  
1 - sigma_1_sq = alpha_1 → signal fraction remaining at t=1. Variance term in KL divergence.  

kl_std_normal(mean_sq, sigma_1_sq) computes KL(𝑁(mean,var)∣∣𝑁(0,1))  
.sum((1,2,3)) → sum over all pixels/channels  
bpd_factor → convert to bits per dimension  


In [ ]:
def kl_std_normal(mean_squared, var):
    return 0.5 * (var + mean_squared - torch.log(var.clamp(min=1e-15)) - 1.0)

In [ ]:
        # *** Latent loss (bpd): KL divergence from N(0, 1) to q(z_1 | x)
        gamma_1 = self.gamma(torch.tensor([1.0], device=self.device))
        sigma_1_sq = sigmoid(gamma_1)
        mean_sq = (1 - sigma_1_sq) * x**2  # (alpha_1 * x)**2
        latent_loss = kl_std_normal(mean_sq, sigma_1_sq).sum((1, 2, 3)) * bpd_factor

##### Reconstruction loss
For each image, look up the model’s predicted probability for the true pixel value at each position, take the log, sum over all pixels, negate it, and normalize to bits-per-dimension. This measures how well the model can reconstruct the original image from the latent.

In [ ]:
def log_probs_x_z0(self, x=None, z_0=None):
        """Computes log p(x | z_0) for all possible values of x.

        Compute p(x_i | z_0i), with i = pixel index, for all possible values of x_i in
        the vocabulary. We approximate this with q(z_0i | x_i). Unnormalized logits are:
            -1/2 SNR_0 (z_0 / alpha_0 - k)^2
        where k takes all possible x_i values. Logits are then normalized to logprobs.

        The method returns a tensor of shape (B, C, H, W, vocab_size) containing, for
        each pixel, the log probabilities for all `vocab_size` possible values of that
        pixel. The output sums to 1 over the last dimension.

        The method accepts either `x` or `z_0` as input. If `z_0` is given, it is used
        directly. If `x` is given, a sample z_0 is drawn from q(z_0 | x). It's more
        efficient to pass `x` directly, if available.

        Args:
            x: Input image, shape (B, C, H, W).
            z_0: z_0 to be decoded, shape (B, C, H, W).

        Returns:
            log_probs: Log probabilities of shape (B, C, H, W, vocab_size).
        """
        gamma_0 = self.gamma(torch.tensor([0.0], device=self.device))
        if x is None and z_0 is not None:
            z_0_rescaled = z_0 / sqrt(sigmoid(-gamma_0))  # z_0 / alpha_0
        elif z_0 is None and x is not None:
            # Equal to z_0/alpha_0 with z_0 sampled from q(z_0 | x)
            z_0_rescaled = x + exp(0.5 * gamma_0) * torch.randn_like(x)  # (B, C, H, W)
        else:
            raise ValueError("Must provide either x or z_0, not both.")
        z_0_rescaled = z_0_rescaled.unsqueeze(-1)  # (B, C, H, W, 1)
        x_lim = 1 - 1 / self.vocab_size
        x_values = linspace(-x_lim, x_lim, self.vocab_size, device=self.device)
        logits = -0.5 * exp(-gamma_0) * (z_0_rescaled - x_values) ** 2  # broadcast x
        log_probs = torch.log_softmax(logits, dim=-1)  # (B, C, H, W, vocab_size)
        return log_probs

In [ ]:
        # *** Reconstruction loss (bpd): - E_{q(z_0 | x)} [log p(x | z_0)].
        # Compute log p(x | z_0) for all possible values of each pixel in x.
        log_probs = self.log_probs_x_z0(x)  # (B, C, H, W, vocab_size)

        # One-hot representation of original image. Shape: (B, C, H, W, vocab_size).
        x_one_hot = torch.zeros((*x.shape, self.vocab_size), device=self.device)
        x_one_hot.scatter_(4, img_int.unsqueeze(-1), 1)  # one-hot over last dim
        # Select the correct log probabilities.
        log_probs = (x_one_hot * log_probs).sum(-1)  # (B, C, H, W)

        # Overall logprob for each image in batch.
        recons_loss = -log_probs.sum((1, 2, 3)) * bpd_factor

In [ ]:
        # *** Overall loss in bpd. Shape (B, ).
        loss = diffusion_loss + latent_loss + recons_loss



        ### RECORDING METRICS
        # with torch.no_grad():
        #     gamma_0 = self.gamma(torch.tensor([0.0], device=self.device))
        # metrics = {
        #     "bpd": loss.mean(),
        #     "diff_loss": diffusion_loss.mean(),
        #     "latent_loss": latent_loss.mean(),
        #     "loss_recon": recons_loss.mean(),
        #     "gamma_0": gamma_0.item(),
        #     "gamma_1": gamma_1.item(),
        # }
        # return loss.mean(), metrics


### THE FINAL CONNECTION
in case you missed it like I did...
#### UNetVDM → the core network:
- Takes a noisy latent z and a noise-level g_t
- Predicts a residual / denoising output
- Implements down path, mid bottleneck, up path, skip connections, attention, conv_out

#### VDM → diffusion process wrapper:
- Handles forward (noising) and backward (denoising) processes
- Computes z_t from x_0 for training (z_t is a random noisy latent)
- Passes z_t and g_t into the UNet for prediction
- Computes the loss for training
- Provides sampling / inference functions

In [ ]:
### in train.py 
model = UNetVDM(cfg)
diffusion = VDM(model, cfg, image_shape=train_set[0][0].shape)